In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp03
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp03/ex_4.py ──
"""
Shared infrastructure for MLFP03 Exercise 4 — Gradient Boosting Deep Dive.

Contains: Singapore credit-scoring data loader, preprocessing pipeline
wrapper, numpy/polars conversion, model-zoo factories (XGBoost/LightGBM/
CatBoost with identical defaults), evaluation metric helpers, and the
output directory used by every technique file.

Technique-specific code (from-scratch boosting loops, split-gain formulas,
hyperparameter sweeps, early-stopping analysis) does NOT belong here — it
lives in the per-technique files under `modules/mlfp03/solutions/ex_4/`.
"""

from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    f1_score,
    log_loss,
    roc_auc_score,
)

from kailash_ml import PreprocessingPipeline
from kailash_ml.interop import to_sklearn_input


# ════════════════════════════════════════════════════════════════════════
# CONFIG — output directory, random seeds, dataset tag
# ════════════════════════════════════════════════════════════════════════

SEED = 42
DATASET_MODULE = "mlfp02"
DATASET_FILE = "sg_credit_scoring.parquet"
TARGET_COLUMN = "default"

OUTPUT_DIR = Path("outputs") / "mlfp03" / "ex_4_boosting"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Singapore credit-scoring, shared across all four techniques
# ════════════════════════════════════════════════════════════════════════


def load_credit_data() -> pl.DataFrame:
    """Load the Singapore credit-scoring dataset via MLFPDataLoader."""
    loader = MLFPDataLoader()
    return loader.load(DATASET_MODULE, DATASET_FILE)


def prepare_credit_split() -> dict[str, Any]:
    """Load credit data and return a train/test split ready for boosting.

    Returns a dict with:
      X_train, y_train, X_test, y_test : numpy arrays
      feature_names                    : list[str]
      default_rate                     : float (positive-class prevalence)

    Tree models do not need normalisation, so we set ``normalize=False``.
    Categoricals are ordinal-encoded because XGBoost/LightGBM expect numeric
    input; CatBoost would accept raw categoricals but we keep the pipeline
    consistent across all three libraries for a fair comparison.
    """
    credit = load_credit_data()

    pipeline = PreprocessingPipeline()
    result = pipeline.setup(
        data=credit,
        target=TARGET_COLUMN,
        train_size=0.8,
        seed=SEED,
        normalize=False,
        categorical_encoding="ordinal",
        imputation_strategy="median",
    )

    feature_cols = [c for c in result.train_data.columns if c != TARGET_COLUMN]
    X_train, y_train, col_info = to_sklearn_input(
        result.train_data,
        feature_columns=feature_cols,
        target_column=TARGET_COLUMN,
    )
    X_test, y_test, _ = to_sklearn_input(
        result.test_data,
        feature_columns=feature_cols,
        target_column=TARGET_COLUMN,
    )

    return {
        "X_train": X_train,
        "y_train": y_train,
        "X_test": X_test,
        "y_test": y_test,
        "feature_names": col_info["feature_columns"],
        "default_rate": float(credit[TARGET_COLUMN].mean()),
    }


# ════════════════════════════════════════════════════════════════════════
# MODEL FACTORIES — identical defaults for fair comparison
# ════════════════════════════════════════════════════════════════════════


def make_xgboost(
    n_estimators: int = 500,
    learning_rate: float = 0.1,
    max_depth: int = 6,
    **extra: Any,
):
    """Build an XGBoost classifier with course-standard defaults."""
    import xgboost as xgb

    return xgb.XGBClassifier(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        eval_metric="logloss",
        random_state=SEED,
        verbosity=0,
        **extra,
    )


def make_lightgbm(
    n_estimators: int = 500,
    learning_rate: float = 0.1,
    max_depth: int = 6,
    **extra: Any,
):
    """Build a LightGBM classifier with course-standard defaults."""
    import lightgbm as lgb

    return lgb.LGBMClassifier(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        num_leaves=31,
        random_state=SEED,
        verbose=-1,
        **extra,
    )


def make_catboost(
    iterations: int = 500,
    learning_rate: float = 0.1,
    depth: int = 6,
    **extra: Any,
):
    """Build a CatBoost classifier with course-standard defaults."""
    import catboost as cb

    return cb.CatBoostClassifier(
        iterations=iterations,
        learning_rate=learning_rate,
        depth=depth,
        random_seed=SEED,
        verbose=0,
        **extra,
    )


# ════════════════════════════════════════════════════════════════════════
# EVALUATION — shared metric helpers for imbalanced binary classification
# ════════════════════════════════════════════════════════════════════════


def evaluate_classifier(
    y_true: np.ndarray,
    y_proba: np.ndarray,
    threshold: float = 0.5,
) -> dict[str, float]:
    """Return the full boosting-eval metric bundle.

    AUC-PR is the primary metric — with a 12% default rate, AUC-ROC rewards
    models that rank negatives correctly even when they miss every default.
    """
    y_pred = (y_proba >= threshold).astype(int)
    return {
        "auc_roc": float(roc_auc_score(y_true, y_proba)),
        "auc_pr": float(average_precision_score(y_true, y_proba)),
        "log_loss": float(log_loss(y_true, y_proba)),
        "brier": float(brier_score_loss(y_true, y_proba)),
        "f1": float(f1_score(y_true, y_pred)),
    }


def print_metrics(
    name: str, metrics: dict[str, float], train_time: float | None = None
) -> None:
    """One-line headline: AUC-ROC | AUC-PR | Log Loss | F1 | Time."""
    time_str = f" | time={train_time:.2f}s" if train_time is not None else ""
    print(
        f"  {name}: "
        f"AUC-ROC={metrics['auc_roc']:.4f} | "
        f"AUC-PR={metrics['auc_pr']:.4f} | "
        f"log_loss={metrics['log_loss']:.4f} | "
        f"F1={metrics['f1']:.4f}"
        f"{time_str}"
    )


# ════════════════════════════════════════════════════════════════════════
# FROM-SCRATCH GRADIENT BOOSTING (used by 01_boosting_theory.py)
# ════════════════════════════════════════════════════════════════════════


def xgb_split_gain(
    g_left: float,
    h_left: float,
    g_right: float,
    h_right: float,
    lambda_reg: float = 1.0,
    gamma: float = 0.0,
) -> float:
    """XGBoost split-gain formula from 2nd-order Taylor expansion of log-loss.

    Gain = ½ [G_L²/(H_L+λ) + G_R²/(H_R+λ) - (G_L+G_R)²/(H_L+H_R+λ)] - γ
    """
    left_score = g_left**2 / (h_left + lambda_reg)
    right_score = g_right**2 / (h_right + lambda_reg)
    parent_score = (g_left + g_right) ** 2 / (h_left + h_right + lambda_reg)
    return 0.5 * (left_score + right_score - parent_score) - gamma


def make_1d_demo(n: int = 200) -> tuple[np.ndarray, np.ndarray]:
    """Generate a 1D logistic-shaped binary classification demo.

    Used by the from-scratch boosting loop to show that residual-fitting
    converges in just a handful of rounds.
    """
    rng = np.random.default_rng(SEED)
    x = rng.uniform(0, 1, n).reshape(-1, 1)
    true_proba = 1 / (1 + np.exp(-10 * (x.ravel() - 0.5)))
    y = rng.binomial(1, true_proba)
    return x, y




# ════════════════════════════════════════════════════════════════════════
# MLFP03 — Exercise 4.3: LightGBM and CatBoost — Same Task, Faster Trees
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Train LightGBM and CatBoost on the same Singapore credit data
#   - Explain LightGBM's histogram-based split finding (why it's fast on
#     large data) and GOSS (gradient-based one-side sampling)
#   - Explain CatBoost's ordered boosting (why it's robust to target
#     leakage on categorical features)
#   - Compare the three libraries on AUC-PR, log loss, and train time
#   - Decide which library to use based on data shape and constraints
#
# PREREQUISITES: Exercise 4.2 (XGBoost baseline on the same dataset).
#
# ESTIMATED TIME: ~35 min
#
# TASKS:
#   1. Theory — how LightGBM and CatBoost differ from XGBoost
#   2. Build — LightGBM + CatBoost with matched hyperparameters
#   3. Train — fit both, record train time and AUC-PR
#   4. Visualise — side-by-side bar chart with XGBoost as baseline
#   5. Apply — Singapore supermarket chain picks a library for fraud
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import time

import plotly.graph_objects as go
from dotenv import load_dotenv


load_dotenv()



## THEORY — Three Libraries, One Gradient-Boosting Recipe


In [ ]:
# All three libraries build additive trees on pseudo-residuals — the
# theory from 4.1 applies identically. They differ in HOW they find the
# best split at each round:
#
# XGBoost: exact split finding. Sorts every feature, evaluates every
#   possible split threshold. Accurate but O(n·d·log n) per round.
#
# LightGBM: histogram-based split finding. Buckets each feature into
#   ~256 bins, evaluates splits between bins only. O(n·d) per round —
#   ~5-10x faster on large data. Adds two more tricks:
#     - GOSS (gradient-based one-side sampling): keep all rows with
#       large gradients (hard examples), randomly sample the easy ones.
#     - EFB (exclusive feature bundling): bundle sparse features that
#       never overlap into a single "super-feature" — further speedup
#       on wide sparse data.
#   LightGBM also grows trees leaf-wise (best-first) instead of
#   level-wise, which usually gives lower loss for the same leaf count
#   but can overfit if max_depth is unset.
#
# CatBoost: ordered boosting. The problem CatBoost solves is target
#   leakage on categorical features — "mean-target encoding" (replace
#   each category with its average target) leaks the target into the
#   training set, inflating training accuracy. CatBoost fixes this by
#   computing target statistics only from rows that come BEFORE the
#   current row in a random permutation, then re-permuting each round.
#   Result: best out-of-the-box performance on data with many high-
#   cardinality categoricals, and the least tuning effort of the three.
#
# Practical decision tree:
#   ≫ >1M rows, mostly numeric        → LightGBM (speed wins)
#   ≫ High-cardinality categoricals   → CatBoost (no manual encoding)
#   ≫ Medium data, mixed types        → XGBoost (stable default)
#
# We train all three with matched hyperparameters so the comparison is
# about the LIBRARY, not the hyperparameters.



## TASK 2 — BUILD all three models with matched defaults


In [ ]:
print("\n" + "=" * 70)
print("  LightGBM + CatBoost vs XGBoost — Singapore Credit Scoring")
print("=" * 70)

data = prepare_credit_split()
X_train, y_train = data["X_train"], data["y_train"]
X_test, y_test = data["X_test"], data["y_test"]

print(f"\n  Train: {X_train.shape} | Test: {X_test.shape}")
print(f"  Default rate: {data['default_rate']:.2%}")

models = {
    "XGBoost": make_xgboost(n_estimators=500, learning_rate=0.1, max_depth=6),
    "LightGBM": make_lightgbm(n_estimators=500, learning_rate=0.1, max_depth=6),
    "CatBoost": make_catboost(iterations=500, learning_rate=0.1, depth=6),
}



## TASK 3 — TRAIN each library and record time + metrics


In [ ]:
results: dict[str, dict] = {}
print("\n  --- Training ---")
for name, model in models.items():
    t0 = time.perf_counter()
    if name == "CatBoost":
        model.fit(X_train, y_train, eval_set=(X_test, y_test))
    else:
        model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    train_time = time.perf_counter() - t0

    y_proba = model.predict_proba(X_test)[:, 1]
    metrics = evaluate_classifier(y_test, y_proba)
    results[name] = {"metrics": metrics, "train_time": train_time, "model": model}
    print_metrics(name, metrics, train_time=train_time)



### Checkpoint 1


In [ ]:
for name, r in results.items():
    assert r["metrics"]["auc_roc"] > 0.7, f"{name} AUC-ROC should exceed 0.7"
    assert r["metrics"]["auc_pr"] > 0.3, f"{name} AUC-PR should exceed 0.3"
# INTERPRETATION: On 5K rows the three libraries are usually within 0.01
# AUC-PR of each other — the data is too small to favour one algorithm.
# The interesting signal here is train time: CatBoost is typically 3-5x
# slower than XGBoost on this size because of the ordered-boosting
# permutation overhead. LightGBM is usually fastest.
print("\n[ok] Checkpoint 1 passed — all three libraries trained on credit data\n")



## TASK 4 — VISUALISE the comparison


In [ ]:
# One chart with two stacked panels: AUC-PR (higher is better) and train
# time (lower is better). Side-by-side bars so the speed/quality trade-
# off jumps out visually.

names = list(results.keys())
auc_pr_values = [results[n]["metrics"]["auc_pr"] for n in names]
auc_roc_values = [results[n]["metrics"]["auc_roc"] for n in names]
log_loss_values = [results[n]["metrics"]["log_loss"] for n in names]
time_values = [results[n]["train_time"] for n in names]

fig = go.Figure()
fig.add_trace(
    go.Bar(name="AUC-PR (higher=better)", x=names, y=auc_pr_values, yaxis="y1")
)
fig.add_trace(
    go.Bar(name="Train time (s, lower=better)", x=names, y=time_values, yaxis="y2")
)
fig.update_layout(
    title="Boosting Library Comparison — Singapore Credit Default",
    barmode="group",
    yaxis=dict(title="AUC-PR", range=[0, max(auc_pr_values) * 1.25]),
    yaxis2=dict(title="Train time (s)", overlaying="y", side="right"),
)
viz_path = OUTPUT_DIR / "ex4_03_library_comparison.html"
fig.write_html(viz_path)
print(f"  Saved: {viz_path}")

# Console table so the reader sees numbers even without opening the HTML
print("\n  --- Library Comparison Table ---")
print(
    f"  {'Library':<10} {'AUC-ROC':>10} {'AUC-PR':>10} {'Log Loss':>10} {'Time (s)':>10}"
)
print("  " + "─" * 56)
for n, auc_pr, auc_roc, ll, t in zip(
    names, auc_pr_values, auc_roc_values, log_loss_values, time_values
):
    print(f"  {n:<10} {auc_roc:>10.4f} {auc_pr:>10.4f} {ll:>10.4f} {t:>10.2f}")



### Checkpoint 2


In [ ]:
best_name = max(results.items(), key=lambda kv: kv[1]["metrics"]["auc_pr"])[0]
fastest_name = min(results.items(), key=lambda kv: kv[1]["train_time"])[0]
assert best_name in results, "best_name must be a trained library"
assert fastest_name in results, "fastest_name must be a trained library"
# INTERPRETATION: Report BOTH numbers ("best by AUC-PR", "fastest"). On a
# 5K-row dataset these are usually different libraries — the fastest is
# often not the most accurate. Knowing both lets you choose: do you pay
# for 0.005 AUC-PR with 3x training time? Depends on how often you
# re-train and how much each default costs.
print("\n[ok] Checkpoint 2 passed — comparison table computed\n")



## TASK 5 — APPLY: FairPrice Fraud Detection Library Choice


In [ ]:
# SCENARIO: NTUC FairPrice (Singapore's largest grocery retailer) runs a
# real-time fraud-detection model on card-present transactions across
# ~160 stores. The model scores every swipe in <50ms and flags suspects
# for a manager callback before the receipt prints.
#
# Data shape:
#   - Rows: ~4M transactions per day
#   - Features: 60% numeric (amount, velocity, time-of-day), 40%
#     categorical (store, SKU bundles, payment network, card BIN)
#   - Cardinality: store has 160 levels; SKU bundle has ~12,000 levels;
#     card BIN has ~9,000 levels
#   - Class balance: 0.2% fraud rate (severely imbalanced)
#   - Re-train cadence: nightly, on the last 30 days = ~120M rows
#
# LIBRARY CHOICE: CatBoost
#   - High-cardinality categoricals (SKU bundle, card BIN) would require
#     ordinal encoding for XGBoost/LightGBM, leaking ordering info into
#     features that have no natural order.
#   - CatBoost's ordered boosting handles them without manual encoding
#     and without target leakage.
#   - The 3-5x train-time penalty vs LightGBM is acceptable: 120M rows
#     trains overnight on a 16-core box (~4 hours), well inside the
#     nightly window.
#
# WHY NOT LIGHTGBM: LightGBM on 160 store levels and 12K SKU bundles
# requires target encoding — this is the exact failure mode CatBoost was
# built for. You can make it work with careful cross-validated target
# encoding, but you own that pipeline and its bugs forever.
#
# WHY NOT XGBOOST: Same high-cardinality encoding problem, plus XGBoost
# is ~5x slower than LightGBM on 120M rows so it would blow through the
# nightly window.
#
# BUSINESS IMPACT: FairPrice processes ~S$4.5B in card transactions per
# year. Industry benchmarks put card-present fraud at 3-6 basis points
# of sales = S$1.3-2.7M in annual losses. A model with AUC-PR 0.65
# (typical for well-tuned CatBoost on this data shape) recovers ~40% of
# fraud = ~S$800K-1.1M per year in avoided losses, against a ~S$80K/year
# total infrastructure cost for training + scoring. 10-14x ROI before
# accounting for the 0.15% lift in legitimate-transaction approval rates
# from fewer false positives — which is usually the larger business win.



## REFLECTION


[x] Trained XGBoost, LightGBM, and CatBoost on the same credit data
      with matched hyperparameters
  [x] Read each library's design in terms of split-finding strategy
      (exact vs histogram vs ordered boosting)
  [x] Compared AUC-PR and train time side-by-side
  [x] Identified {best_name} as the best on this dataset by AUC-PR
  [x] Identified {fastest_name} as the fastest to train
  [x] Matched library choice to data shape using the FairPrice scenario

  KEY INSIGHT: The three libraries are not interchangeable. LightGBM is
  the speed winner on large numeric data; CatBoost is the accuracy
  winner on high-cardinality categorical data; XGBoost is the stable
  default when the data is mixed and medium-sized. Pick before you
  tune — tuning cannot rescue a library mismatch.

  Next: 04_boosting_tuning.py — hyperparameter sweeps, learning-rate
  sensitivity, and early stopping on the same dataset.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

